# Baseline QGRU — hyperparameter tuning

Sweeps the vom-Scheidt search axes (recurrent size × dense size × learning rate) with
values adapted to this project's 456-output monotone head, selects on **validation
pinball**, and saves the single best early-stopped model as a rebuildable checkpoint.

**Runs from the same folder as `forecasting.py`** (imports the tested core from it).

Decisions baked in (from the pipeline contract):
- split: **year / half-year / year** (train 2018 · val 2019-H1 · test 2019-H2→2020-H1)
- **all-hours augmented** origins for the baseline (`gate_aligned_only=False`)
- scaler fit on the **train span only**; validation-based selection; **test never touched here**
- pinball loss = **sum over K and Q, averaged per batch** (same reduction for train, val, and the DFL warm-start)
- early stopping: **patience 10 · max 200 epochs · restore best weights**
- searched: `gru_hidden ∈ {4,8,16,32}`, `dense ∈ {8,16,32,64}`, `lr ∈ {0.001,0.005,0.01}` (36 configs)
- fixed (not searched): dropout 0.1 · weight_decay 1e-4 · 2 GRU layers · eps 1e-4 · batch 64


## 1 · Imports & reproducibility

In [ ]:
import copy
import time
import random
import subprocess

import numpy as np
import pandas as pd
import torch
from pyprojroot import here
import matplotlib.pyplot as plt

# tested core — same directory
from forecasting import (
    QUANTILE_LEVELS, HIST_COLS, FEAT_COLS, EXO_COLS,
    TRAIN_START, VAL_START, TEST_START, TEST_END,
    build_features, reindex_and_impute, make_windows,
    fit_scalers, normalise_hist, normalise_y, denormalise_y,
    Baseline_Forecaster, pinball_loss,
)


In [2]:
# ---- reproducibility (record SEED in the checkpoint) ----
SEED = 20240801
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| seed:", SEED)


device: cpu | seed: 20240801


## 2 · Search grid & fixed hyperparameters

In [ ]:
# ---- searched axes (vom-Scheidt structure; capacities/LR adapted to the 456-output head) ----
GRU_HIDDEN_GRID  = [4, 8, 16, 32]
DENSE_WIDTH_GRID = [8, 16, 32, 64]
LR_GRID          = [0.001, 0.005, 0.01]
GRID = [
    {"gru_hidden_size": g, "dense_hidden_width": d, "lr": lr}
    for g in GRU_HIDDEN_GRID
    for d in DENSE_WIDTH_GRID
    for lr in LR_GRID
]
print(f"{len(GRID)} configs in the sweep")

# ---- fixed across EVERY config (no hidden degrees of freedom in the search) ----
DROPOUT      = 0.1
WEIGHT_DECAY = 1e-4
N_GRU_LAYERS = 2
EPS          = 1e-4
HORIZON      = 24
BATCH_SIZE   = 64

# ---- early stopping ----
MAX_EPOCHS = 200
PATIENCE   = 10     # epochs with no val improvement before stopping
MIN_DELTA  = 0.0    # strict improvement required
# restore-best-weights is implemented inside train_one_config().

# pinball levels on-device; same tensor used for train and val loss
LEVELS = torch.as_tensor(QUANTILE_LEVELS, dtype=torch.float32, device=DEVICE)


48 configs in the sweep


## 3 · Load data → UTC hourly grid → features

Same preprocessing path as `forecasting.py`'s `__main__` (tz-localise → reindex+impute → build features).

In [ ]:
ROOT_DIR = here()
DATA_DIR = ROOT_DIR / "1_data" / "processed"
base_data = pd.read_csv(DATA_DIR / "df_full.csv", parse_dates=["datetime"])
base_data.set_index("datetime", inplace=True)

if base_data.index.tz is None:
    base_data = base_data.tz_localize("UTC")
else:
    base_data = base_data.tz_convert("UTC")
base_data = base_data.sort_index()

base_data  = reindex_and_impute(base_data, HIST_COLS, freq="1h", warn_gap=6)
frame_full = build_features(base_data, feature_cols=FEAT_COLS)
print("frame_full:", frame_full.shape, "|", frame_full.index.min(), "->", frame_full.index.max())


## 4 · Windowing — train & val only

Test split is **deliberately not built here** so tuning cannot see it. History spills back before each split's start for burn-in; the target-based `y_range` keeps split targets disjoint.

In [ ]:
h = pd.Timedelta(hours=1)

Xtr = make_windows(
    frame_full, y_range=(TRAIN_START, VAL_START - h),
    gate_aligned_only=False, issue_hour=9,
    exo_cols=EXO_COLS, hist_cols=HIST_COLS, target_col="prosumption",
)
Xva = make_windows(
    frame_full, y_range=(VAL_START, TEST_START - h),
    gate_aligned_only=False, issue_hour=9,
    exo_cols=EXO_COLS, hist_cols=HIST_COLS, target_col="prosumption",
)

# seam + shape invariants
assert Xtr.de == VAL_START - h, f"train delivery end {Xtr.de} != {VAL_START - h}"
assert Xtr.x_hist.shape[1:] == (168, len(HIST_COLS))
assert Xtr.x_fut.shape[1:]  == (24, len(EXO_COLS))
assert Xtr.y.shape[1:]      == (24,)

print(f"train windows: {Xtr.x_hist.shape[0]:>6}  ({Xtr.ds} -> {Xtr.de})")
print(f"val   windows: {Xva.x_hist.shape[0]:>6}  ({Xva.ds} -> {Xva.de})")


## 5 · Scale (train-only stats) → tensors

In [ ]:
# scaler fit on the TRAIN span ONLY, applied to val (no leakage)
sc = fit_scalers(frame_full.loc[Xtr.hs:Xtr.de], HIST_COLS)
print("mu_y:", round(sc["mu_y"], 3), "| sd_y:", round(sc["sd_y"], 3))

# normalise necessary data (historical and y-value)
xh_tr = normalise_hist(Xtr.x_hist, sc); y_tr = normalise_y(Xtr.y, sc)
xh_va = normalise_hist(Xva.x_hist, sc); y_va = normalise_y(Xva.y, sc)

def _t(a):
    return torch.as_tensor(a, dtype=torch.float32, device=DEVICE)

data = {
    "xh_tr": _t(xh_tr), "xf_tr": _t(Xtr.x_fut), "y_tr": _t(y_tr),
    "xh_va": _t(xh_va), "xf_va": _t(Xva.x_fut), "y_va": _t(y_va),
    "num_hist": Xtr.x_hist.shape[2],
    "num_exo":  Xtr.x_fut.shape[2],
}
print("tensor shapes:", {k: tuple(v.shape) for k, v in data.items() if torch.is_tensor(v)})


## 6 · Train one config, with early stopping + restore-best

In [ ]:
@torch.no_grad()
def val_pinball(model):
    model.eval()
    return pinball_loss(data["y_va"], model(data["xh_va"], data["xf_va"]), LEVELS).item()


def train_one_config(cfg, seed=SEED, verbose=False):
    # fresh, identically-seeded init + optimizer per config for a fair comparison
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = Baseline_Forecaster(
        n_hist_features=data["num_hist"],
        n_exo_features=data["num_exo"],
        n_quantiles=len(QUANTILE_LEVELS),
        gru_hidden_size=cfg["gru_hidden_size"],
        dense_hidden_width=cfg["dense_hidden_width"],
        dropout=DROPOUT, horizon=HORIZON,
        n_gru_layers=N_GRU_LAYERS, eps=EPS,
    ).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=WEIGHT_DECAY)

    n = data["xh_tr"].shape[0] # number of training windows
    best_val, best_epoch, best_state = float("inf"), -1, None
    stale = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, BATCH_SIZE):
            b = perm[i:i + BATCH_SIZE]
            opt.zero_grad()
            loss = pinball_loss(
                data["y_tr"][b], model(data["xh_tr"][b], data["xf_tr"][b]), LEVELS
            )
            loss.backward()
            opt.step()

        v = val_pinball(model)
        if v < best_val - MIN_DELTA:
            best_val, best_epoch = v, epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                break
        if verbose:
            print(f"  epoch {epoch:3d} | val {v:.4f} | best {best_val:.4f}@{best_epoch}")

    model.load_state_dict(best_state)   # restore best weights
    return model, best_val, best_epoch


## 7 · Run the sweep

36 configs. Selection is purely on validation pinball; the best model's weights are kept in memory for the checkpoint.

In [ ]:
results = []
best_overall = {"val": float("inf")}
t0 = time.time()

for j, cfg in enumerate(GRID, 1):
    model, best_val, best_epoch = train_one_config(cfg, seed = SEED, verbose=True)
    results.append({**cfg, "val_pinball": best_val, "best_epoch": best_epoch})
    print(f"[{j:>2}/{len(GRID)}] gru={cfg['gru_hidden_size']:>2} "
          f"dense={cfg['dense_hidden_width']:>2} lr={cfg['lr']:<5} "
          f"-> val {best_val:.4f} @ epoch {best_epoch}")
    if best_val < best_overall["val"]:
        best_overall = {
            "val": best_val, "epoch": best_epoch, "cfg": cfg,
            "state": copy.deepcopy(model.state_dict()),
        }

print(f"\nsweep done in {time.time() - t0:.0f}s | "
      f"best val {best_overall['val']:.4f} with {best_overall['cfg']}")


## 8 · Results table

In [ ]:
res_df = pd.DataFrame(results).sort_values("val_pinball").reset_index(drop=True)
res_df


## 9 · Save best checkpoint

Bundle stores everything the DFL warm-start needs: rebuildable `model_config`, the **train-only scaler**, quantile levels, split dates, seed, git sha, and the full grid for the write-up. Saves a `state_dict` (not a pickled module) so it survives refactors and torch upgrades.

In [ ]:
best_cfg = best_overall["cfg"]
model_config = {
    "n_hist_features":    data["num_hist"],
    "n_exo_features":     data["num_exo"],
    "n_quantiles":        len(QUANTILE_LEVELS),
    "gru_hidden_size":    best_cfg["gru_hidden_size"],
    "dense_hidden_width": best_cfg["dense_hidden_width"],
    "dropout":            DROPOUT,
    "horizon":            HORIZON,
    "n_gru_layers":       N_GRU_LAYERS,
    "eps":                EPS,
}

try:
    git_sha = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=str(here())
    ).decode().strip()
except Exception:
    git_sha = None

OUT_DIR = here() / "4_forecasting"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ckpt_path = OUT_DIR / "baseline_forecaster_best.pt"

torch.save({
    "state_dict":      best_overall["state"],
    "model_config":    model_config,
    "best_lr":         best_cfg["lr"],
    "scaler_stats":    sc,
    "quantile_levels": QUANTILE_LEVELS,
    "split_dates": {
        "train": (str(TRAIN_START), str(VAL_START - h)),
        "val":   (str(VAL_START),   str(TEST_START - h)),
        "test":  (str(TEST_START),  str(TEST_END)),
    },
    "fixed_hparams": {
        "dropout": DROPOUT, "weight_decay": WEIGHT_DECAY,
        "n_gru_layers": N_GRU_LAYERS, "eps": EPS, "batch_size": BATCH_SIZE,
    },
    "early_stopping": {"max_epochs": MAX_EPOCHS, "patience": PATIENCE, "min_delta": MIN_DELTA},
    "val_pinball":  best_overall["val"],
    "best_epoch":   best_overall["epoch"],
    "seed":         SEED,
    "git_sha":      git_sha,
    "grid_results": results,
}, ckpt_path)

print("saved:", ckpt_path)
print("best config:", best_cfg, "| val pinball:", round(best_overall["val"], 4),
      "| epoch:", best_overall["epoch"])


In [ ]:
# --- evaluate: monotonicity ------------------------------------- (cell 22) --
model.eval()
with torch.no_grad():
    q_va = model(data["xh_va"], data["xf_va"])  # (N, H, Q), standardised
diffs = torch.diff(q_va, dim=-1)
print("monotone (all increments > 0):", bool((diffs > 0).all()))
print("min increment:", float(diffs.min()))

q_mw = denormalise_y(torch.Tensor.cpu(q_va), sc)
assert np.all(np.diff(q_mw.numpy(), axis=-1) >= 0), (
    "physical-unit quantiles must stay ordered"
)
print("monotone after inverse-standardise: True")

# --- evaluate: coverage ----------------------------------------- (cell 23) --
with torch.no_grad():
    cover = [
        (data["y_va"].unsqueeze(-1) <= q_va[..., k : k + 1]).float().mean().item()
        for k in range(len(QUANTILE_LEVELS))
    ]
for a, c in zip(QUANTILE_LEVELS.tolist(), cover):
    print(f"level {a:.2f}:  empirical coverage {c:.3f}")
max_dev = max(abs(a - c) for a, c in zip(QUANTILE_LEVELS.tolist(), cover))
print(f"\nmax |coverage - nominal| = {max_dev:.3f}  (small = well calibrated)")

# --- evaluate: forecast plot ------------------------------------ (cell 24) --
Yva = Xva.y[10:20].flatten()
hrs = np.arange(24 * 10)
qp = q_mw.numpy()[10:20].reshape(-1, 19)  # (H, Q) physical units
mid = len(QUANTILE_LEVELS) // 2
plt.figure(figsize=(9, 4))
for k in range(mid):  # nested prediction bands
    plt.fill_between(hrs, qp[:, k], qp[:, -1 - k], alpha=0.15, color="C0")
plt.plot(hrs, qp[:, mid], color="C0", label="median forecast")
plt.plot(hrs, Yva, "k--", label="realised")
plt.xlabel("hour of horizon")
plt.ylabel("prosumption (MW)")
plt.title("Monotone quantile forecast — ten validation days")
plt.legend()
plt.tight_layout()
plt.show()